<a href="https://colab.research.google.com/github/fatmasenguler/Mutation_KRAS_analysis/blob/main/Table_1_Global_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install biopython networkx pandas matplotlib numpy

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
KRAS Table-1 global thermodynamics
==================================

Exact spanning-tree thermodynamics with a controlled finite-difference audit.

Main corrections relative to the earlier implementation
---------------------------------------------------------
1. Chain A is selected explicitly; the first chain is not assumed to be A.
2. Only standard ATOM residues containing a C-alpha atom are included.
3. Coordinates and all matrix calculations use float64.
4. Residue identifiers retain insertion codes.
5. Graph connectedness and residue-ID uniqueness are checked explicitly.
6. The same grounded node is removed in exact and finite-difference calculations.
7. log Z is always evaluated by numpy.linalg.slogdet.
8. No arbitrary eigenvalue threshold is used.
9. Forward/reverse edge summation order is audited at fixed h.
10. Exact heat capacity is evaluated from beta-derivative trace identities;
    no finite-difference step enters the exact result.

Weights
-------
    w_e(beta) = exp(-beta d_e),    beta = 1/kT

For a grounded Laplacian L:

    d ln Z / d beta
        = Tr(L^{-1} L_beta)

    d² ln Z / d beta²
        = Tr(L^{-1} L_beta_beta)
          - Tr[(L^{-1} L_beta)²]

Therefore:

    <E>    = -d ln Z / d beta
    Var(E) =  d² ln Z / d beta²
    C      = Var(E) / (kT)²
    F      = -kT ln Z
    S      = (<E> - F) / kT

Finite-difference audit
-----------------------
With tau = kT and f(tau) = ln Z(tau):

    C = 2 tau f'(tau) + tau² f''(tau)

The finite-difference calculation is included only as a numerical audit.

Installation
------------
Local terminal:
    pip install numpy networkx biopython

Google Colab:
    !pip install numpy networkx biopython

Cutoff
------
The default cutoff below is 8.0 Angstrom because the reported Table-1
energy and entropy appear to correspond to 8.0 Angstrom.

For the 7.8 Angstrom graph stated in the manuscript text or used in path
calculations, change:

    CUTOFF = 7.8
"""

from pathlib import Path
from urllib.request import urlretrieve
import warnings

import numpy as np
import networkx as nx
from Bio.PDB import PDBParser


# ============================================================================
# USER SETTINGS
# ============================================================================

WT_PDB_ID = "6GOD"
MUT_PDB_ID = "6GOF"

WT_PDB_FILE = "6GOD.pdb"
MUT_PDB_FILE = "6GOF.pdb"

CHAIN_ID = "A"

# Change to 7.8 to audit the cutoff stated elsewhere in the manuscript.
CUTOFF = 7.8

KT = 1.0

# Same grounded matrix index is used for exact and FD calculations.
GROUND_INDEX = 0

# Download the PDB files from RCSB if they are not in the working directory.
AUTO_DOWNLOAD = True

FD_STEPS = (
    5.0e-2,
    2.0e-2,
    1.0e-2,
    5.0e-3,
    1.0e-3,
    5.0e-4,
    1.0e-4,
    5.0e-5,
    1.0e-5,
)

EDGE_ORDER_AUDIT_H = 1.0e-4


# ============================================================================
# FILE HANDLING
# ============================================================================

def ensure_pdb_file(pdb_id, filename, auto_download=True):
    """
    Return a local PDB path.

    If the file is missing and auto_download=True, download it from RCSB.
    """
    path = Path(filename)

    if path.is_file() and path.stat().st_size > 0:
        return str(path)

    if not auto_download:
        raise FileNotFoundError(
            f"{filename} was not found. Place it in the working directory."
        )

    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"

    print(f"Downloading {pdb_id.upper()} from RCSB...")
    try:
        urlretrieve(url, path)
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {pdb_id} from {url}. "
            f"Download {filename} manually and rerun."
        ) from exc

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Downloaded file {filename} is empty.")

    return str(path)


# ============================================================================
# PDB -> C-ALPHA CONTACT GRAPH
# ============================================================================

def format_residue_id(residue_id):
    """Convert a residue identifier tuple to a readable label."""
    resseq, icode = residue_id
    return f"{resseq}{icode}" if icode else str(resseq)


def build_ca_graph(pdb_file, chain_id="A", cutoff=8.0):
    """
    Build the weighted C-alpha contact graph.

    Nodes
    -----
    Residues in the requested chain containing a C-alpha atom.

    Node identifier
    ---------------
    (author_residue_number, insertion_code)

    Edge criterion
    --------------
    Euclidean C-alpha distance <= cutoff.

    Stored edge attribute
    ---------------------
    distance : C-alpha distance in Angstrom
    weight   : same distance, retained for compatibility
    """
    if cutoff <= 0:
        raise ValueError("cutoff must be positive.")

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(pdb_file).stem, pdb_file)

    try:
        model = next(structure.get_models())
    except StopIteration as exc:
        raise ValueError(f"No model was found in {pdb_file}.") from exc

    if not model.has_id(chain_id):
        available = [chain.id for chain in model.get_chains()]
        raise ValueError(
            f"Chain {chain_id!r} was not found in {pdb_file}. "
            f"Available chains: {available}"
        )

    chain = model[chain_id]

    coords = []
    residue_ids = []

    for residue in chain:
        hetflag, resseq, insertion_code = residue.get_id()

        # Match the ordinary protein-residue construction:
        # ignore water, ligands and other HETATM residues.
        if hetflag != " ":
            continue

        if "CA" not in residue:
            continue

        residue_id = (int(resseq), str(insertion_code).strip())

        if residue_id in residue_ids:
            raise ValueError(
                f"Duplicate residue identifier "
                f"{format_residue_id(residue_id)} in {pdb_file}."
            )

        coordinate = np.asarray(
            residue["CA"].get_coord(),
            dtype=np.float64,
        )

        if coordinate.shape != (3,) or not np.all(np.isfinite(coordinate)):
            raise ValueError(
                f"Invalid C-alpha coordinate for residue "
                f"{format_residue_id(residue_id)} in {pdb_file}."
            )

        residue_ids.append(residue_id)
        coords.append(coordinate)

    if len(residue_ids) < 2:
        raise ValueError(
            f"Fewer than two C-alpha residues were found in chain "
            f"{chain_id} of {pdb_file}."
        )

    if len(residue_ids) != len(set(residue_ids)):
        raise ValueError("Residue identifiers are not unique.")

    coordinates = np.vstack(coords).astype(np.float64, copy=False)
    n = len(residue_ids)

    graph = nx.Graph()
    graph.add_nodes_from(residue_ids)

    for i in range(n - 1):
        delta = coordinates[i + 1:] - coordinates[i]
        distances = np.linalg.norm(delta, axis=1)

        contact_offsets = np.flatnonzero(distances <= cutoff)

        for offset in contact_offsets:
            j = i + 1 + int(offset)
            distance = float(distances[offset])

            graph.add_edge(
                residue_ids[i],
                residue_ids[j],
                distance=distance,
                weight=distance,
            )

    if graph.number_of_edges() == 0:
        raise ValueError(
            f"No contacts were found at cutoff {cutoff:.3f} Angstrom."
        )

    if not nx.is_connected(graph):
        components = sorted(
            (len(component) for component in nx.connected_components(graph)),
            reverse=True,
        )
        raise ValueError(
            f"The contact graph for {pdb_file} is disconnected at "
            f"cutoff {cutoff:.3f} Angstrom. "
            f"Component sizes: {components}"
        )

    return graph, residue_ids, coordinates


# ============================================================================
# MATRIX UTILITIES
# ============================================================================

def reduced_matrix(matrix, ground_index=0):
    """Delete one common row and column from a Laplacian-like matrix."""
    matrix = np.asarray(matrix, dtype=np.float64)

    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("matrix must be square.")

    n = matrix.shape[0]

    if n < 2:
        raise ValueError("matrix must have dimension at least 2.")

    if not 0 <= ground_index < n:
        raise ValueError(
            f"ground_index={ground_index} is invalid for a {n}x{n} matrix."
        )

    keep = np.ones(n, dtype=bool)
    keep[ground_index] = False

    return matrix[np.ix_(keep, keep)]


def check_symmetric(matrix, name, tolerance=1.0e-11):
    """Check numerical symmetry."""
    error = np.max(np.abs(matrix - matrix.T))
    scale = max(1.0, np.max(np.abs(matrix)))

    if error > tolerance * scale:
        raise ValueError(
            f"{name} is not numerically symmetric: max error={error:.3e}"
        )


def logdet_positive_definite(matrix, name="matrix"):
    """
    Compute log(det(matrix)) without forming the determinant.

    A valid grounded Laplacian must have positive determinant.
    """
    sign, logdet = np.linalg.slogdet(matrix)

    if sign <= 0 or not np.isfinite(logdet):
        raise ValueError(
            f"{name} does not have a valid positive finite determinant: "
            f"sign={sign}, logdet={logdet}"
        )

    return float(logdet)


# ============================================================================
# LAPLACIAN CONSTRUCTION
# ============================================================================

def add_weighted_edge(matrix, i, j, value):
    """Add value * b_ij b_ij^T to a Laplacian matrix."""
    matrix[i, i] += value
    matrix[j, j] += value
    matrix[i, j] -= value
    matrix[j, i] -= value


def beta_laplacian_derivatives(graph, residue_ids, kT):
    """
    Construct L, dL/d beta and d²L/d beta².

    For each edge:

        w(beta)   = exp(-beta d)
        w'(beta)  = -d exp(-beta d)
        w''(beta) = d² exp(-beta d)
    """
    if kT <= 0:
        raise ValueError("kT must be positive.")

    n = len(residue_ids)
    index = {residue: i for i, residue in enumerate(residue_ids)}

    if len(index) != n:
        raise ValueError("Residue identifiers are not unique.")

    beta = 1.0 / float(kT)

    L0 = np.zeros((n, n), dtype=np.float64)
    L1 = np.zeros((n, n), dtype=np.float64)
    L2 = np.zeros((n, n), dtype=np.float64)

    for u, v, data in graph.edges(data=True):
        distance = float(data["distance"])

        weight = np.exp(-beta * distance)
        weight_beta = -distance * weight
        weight_beta_beta = distance * distance * weight

        i = index[u]
        j = index[v]

        add_weighted_edge(L0, i, j, weight)
        add_weighted_edge(L1, i, j, weight_beta)
        add_weighted_edge(L2, i, j, weight_beta_beta)

    check_symmetric(L0, "L0")
    check_symmetric(L1, "L1")
    check_symmetric(L2, "L2")

    return L0, L1, L2


def laplacian_at_tau(
    graph,
    residue_ids,
    tau,
    reverse_edge_order=False,
):
    """
    Build L(tau) using w_e = exp(-d_e/tau).

    Reversing the edge order leaves the mathematical Laplacian unchanged,
    but may alter its last floating-point bits because floating-point
    addition is not associative.
    """
    if tau <= 0:
        raise ValueError("tau must be positive.")

    n = len(residue_ids)
    index = {residue: i for i, residue in enumerate(residue_ids)}

    if len(index) != n:
        raise ValueError("Residue identifiers are not unique.")

    edges = list(graph.edges(data=True))

    if reverse_edge_order:
        edges.reverse()

    L = np.zeros((n, n), dtype=np.float64)

    for u, v, data in edges:
        distance = float(data["distance"])
        weight = np.exp(-distance / tau)

        i = index[u]
        j = index[v]

        add_weighted_edge(L, i, j, weight)

    check_symmetric(L, "L(tau)")

    return L


# ============================================================================
# EXACT THERMODYNAMICS
# ============================================================================

def global_thermo_exact(
    graph,
    residue_ids,
    kT=1.0,
    ground_index=0,
):
    """
    Compute exact global thermodynamics from trace identities.

    No finite-difference step is used.
    """
    L0, L1, L2 = beta_laplacian_derivatives(
        graph,
        residue_ids,
        kT,
    )

    L0r = reduced_matrix(L0, ground_index)
    L1r = reduced_matrix(L1, ground_index)
    L2r = reduced_matrix(L2, ground_index)

    logZ = logdet_positive_definite(
        L0r,
        name="grounded weighted Laplacian",
    )

    # A1 = L^{-1} L_beta
    # A2 = L^{-1} L_beta_beta
    A1 = np.linalg.solve(L0r, L1r)
    A2 = np.linalg.solve(L0r, L2r)

    trace_A1 = float(np.trace(A1))
    trace_A2 = float(np.trace(A2))

    # Tr(A1 @ A1) = sum_ij A1_ij A1_ji.
    trace_A1_squared = float(
        np.einsum("ij,ji->", A1, A1, optimize=True)
    )

    energy_mean = -trace_A1
    variance_energy = trace_A2 - trace_A1_squared

    # Permit only negligible negative roundoff.
    variance_scale = max(
        1.0,
        abs(trace_A2),
        abs(trace_A1_squared),
    )
    negative_tolerance = 1.0e-10 * variance_scale

    if variance_energy < -negative_tolerance:
        raise ArithmeticError(
            f"Computed energy variance is significantly negative: "
            f"{variance_energy:.12e}"
        )

    if variance_energy < 0:
        warnings.warn(
            "A tiny negative energy variance caused by roundoff was clipped "
            "to zero.",
            RuntimeWarning,
        )
        variance_energy = 0.0

    free_energy = -kT * logZ
    heat_capacity = variance_energy / (kT * kT)
    entropy = (energy_mean - free_energy) / kT

    condition_number = float(np.linalg.cond(L0r))

    return {
        "F": float(free_energy),
        "E_mean": float(energy_mean),
        "S": float(entropy),
        "C": float(heat_capacity),
        "Var_E": float(variance_energy),
        "lnZ": float(logZ),
        "condition_number": condition_number,
        "N": len(residue_ids),
        "E_edges": graph.number_of_edges(),
        "ground_index": ground_index,
        "ground_residue": residue_ids[ground_index],
    }


# ============================================================================
# FINITE-DIFFERENCE AUDIT
# ============================================================================

def logZ_tau(
    graph,
    residue_ids,
    tau,
    ground_index=0,
    reverse_edge_order=False,
):
    """Compute ln Z(tau) by slogdet of the same grounded cofactor."""
    L = laplacian_at_tau(
        graph,
        residue_ids,
        tau,
        reverse_edge_order=reverse_edge_order,
    )

    Lr = reduced_matrix(L, ground_index)

    return logdet_positive_definite(
        Lr,
        name=f"grounded Laplacian at tau={tau:.12g}",
    )


def heat_capacity_tau_fd(
    graph,
    residue_ids,
    kT,
    h,
    ground_index=0,
    reverse_edge_order=False,
):
    """
    Central finite-difference estimate of heat capacity.

        f(tau) = ln Z(tau)

        f'(tau)  ~= [f(tau+h)-f(tau-h)]/(2h)

        f''(tau) ~= [f(tau+h)-2f(tau)+f(tau-h)]/h²

        C = 2 tau f'(tau) + tau² f''(tau)
    """
    if kT <= 0:
        raise ValueError("kT must be positive.")

    if h <= 0:
        raise ValueError("h must be positive.")

    if kT - h <= 0:
        raise ValueError(
            f"kT-h must be positive; received kT={kT}, h={h}."
        )

    f_minus = logZ_tau(
        graph,
        residue_ids,
        kT - h,
        ground_index,
        reverse_edge_order,
    )

    f_zero = logZ_tau(
        graph,
        residue_ids,
        kT,
        ground_index,
        reverse_edge_order,
    )

    f_plus = logZ_tau(
        graph,
        residue_ids,
        kT + h,
        ground_index,
        reverse_edge_order,
    )

    first_derivative = (f_plus - f_minus) / (2.0 * h)

    second_numerator = f_plus - 2.0 * f_zero + f_minus
    second_derivative = second_numerator / (h * h)

    heat_capacity = (
        2.0 * kT * first_derivative
        + kT * kT * second_derivative
    )

    return {
        "C": float(heat_capacity),
        "df": float(first_derivative),
        "ddf": float(second_derivative),
        "second_numerator": float(second_numerator),
        "lnZ_minus": float(f_minus),
        "lnZ_zero": float(f_zero),
        "lnZ_plus": float(f_plus),
    }


# ============================================================================
# AUDIT UTILITIES
# ============================================================================

def percent_change(reference, new_value):
    """Return 100*(new-reference)/reference."""
    if reference == 0:
        return np.nan

    return 100.0 * (new_value - reference) / reference


def matrix_edge_order_difference(
    graph,
    residue_ids,
    tau,
):
    """
    Quantify the matrix-level difference caused only by edge summation order.
    """
    forward = laplacian_at_tau(
        graph,
        residue_ids,
        tau,
        reverse_edge_order=False,
    )

    reverse = laplacian_at_tau(
        graph,
        residue_ids,
        tau,
        reverse_edge_order=True,
    )

    difference = reverse - forward

    return {
        "max_abs": float(np.max(np.abs(difference))),
        "frobenius": float(np.linalg.norm(difference, ord="fro")),
    }


def print_graph_summary(label, graph, residue_ids, cutoff):
    """Print graph diagnostics."""
    first_residue = format_residue_id(residue_ids[0])
    last_residue = format_residue_id(residue_ids[-1])

    print(
        f"{label}: N={len(residue_ids)}, "
        f"edges={graph.number_of_edges()}, "
        f"connected={nx.is_connected(graph)}, "
        f"residue range={first_residue}-{last_residue}, "
        f"cutoff={cutoff:.3f} A"
    )


def print_exact_table(wt, mutant):
    """Print exact thermodynamic results."""
    delta_e = percent_change(wt["E_mean"], mutant["E_mean"])
    delta_s = percent_change(wt["S"], mutant["S"])
    delta_c = percent_change(wt["C"], mutant["C"])

    print("\nEXACT TRACE-FORMULA THERMODYNAMICS")
    print("-" * 70)
    print(
        f"{'Property':<18}"
        f"{'WT':>14}"
        f"{'G12D':>14}"
        f"{'Change (%)':>16}"
    )
    print("-" * 70)

    print(
        f"{'ln Z':<18}"
        f"{wt['lnZ']:>14.6f}"
        f"{mutant['lnZ']:>14.6f}"
        f"{percent_change(wt['lnZ'], mutant['lnZ']):>+16.6f}"
    )

    print(
        f"{'Free energy F':<18}"
        f"{wt['F']:>14.6f}"
        f"{mutant['F']:>14.6f}"
        f"{percent_change(wt['F'], mutant['F']):>+16.6f}"
    )

    print(
        f"{'Energy <E>':<18}"
        f"{wt['E_mean']:>14.6f}"
        f"{mutant['E_mean']:>14.6f}"
        f"{delta_e:>+16.6f}"
    )

    print(
        f"{'Entropy S':<18}"
        f"{wt['S']:>14.6f}"
        f"{mutant['S']:>14.6f}"
        f"{delta_s:>+16.6f}"
    )

    print(
        f"{'Var(E)':<18}"
        f"{wt['Var_E']:>14.6f}"
        f"{mutant['Var_E']:>14.6f}"
        f"{percent_change(wt['Var_E'], mutant['Var_E']):>+16.6f}"
    )

    print(
        f"{'Heat capacity C':<18}"
        f"{wt['C']:>14.6f}"
        f"{mutant['C']:>14.6f}"
        f"{delta_c:>+16.6f}"
    )

    print("-" * 70)

    print(
        f"Condition numbers: "
        f"WT={wt['condition_number']:.6e}, "
        f"G12D={mutant['condition_number']:.6e}"
    )

    print(
        f"Common grounded index: {wt['ground_index']} "
        f"(WT residue {format_residue_id(wt['ground_residue'])}, "
        f"G12D residue {format_residue_id(mutant['ground_residue'])})"
    )


def print_fd_audit(
    wt_graph,
    wt_residues,
    mut_graph,
    mut_residues,
    wt_exact,
    mut_exact,
    kT,
    ground_index,
    steps,
):
    """Print finite-difference heat capacity across step sizes."""
    print("\nFINITE-DIFFERENCE AUDIT")
    print("-" * 84)
    print(
        f"{'h':>12}"
        f"{'C_WT FD':>15}"
        f"{'WT error':>15}"
        f"{'C_G12D FD':>15}"
        f"{'G12D error':>15}"
    )
    print("-" * 84)

    for h in steps:
        wt_fd = heat_capacity_tau_fd(
            wt_graph,
            wt_residues,
            kT,
            h,
            ground_index=ground_index,
            reverse_edge_order=False,
        )

        mut_fd = heat_capacity_tau_fd(
            mut_graph,
            mut_residues,
            kT,
            h,
            ground_index=ground_index,
            reverse_edge_order=False,
        )

        wt_error = wt_fd["C"] - wt_exact["C"]
        mut_error = mut_fd["C"] - mut_exact["C"]

        marker = "  <- manuscript h" if np.isclose(h, 1.0e-4) else ""

        print(
            f"{h:>12.1e}"
            f"{wt_fd['C']:>15.6f}"
            f"{wt_error:>+15.6f}"
            f"{mut_fd['C']:>15.6f}"
            f"{mut_error:>+15.6f}"
            f"{marker}"
        )

    print("-" * 84)

    print(
        f"{'EXACT':>12}"
        f"{wt_exact['C']:>15.6f}"
        f"{0.0:>+15.6f}"
        f"{mut_exact['C']:>15.6f}"
        f"{0.0:>+15.6f}"
    )


def print_edge_order_audit(
    label,
    graph,
    residue_ids,
    exact_result,
    kT,
    h,
    ground_index,
):
    """Audit dependence on floating-point edge summation order."""
    forward = heat_capacity_tau_fd(
        graph,
        residue_ids,
        kT,
        h,
        ground_index=ground_index,
        reverse_edge_order=False,
    )

    reverse = heat_capacity_tau_fd(
        graph,
        residue_ids,
        kT,
        h,
        ground_index=ground_index,
        reverse_edge_order=True,
    )

    matrix_difference = matrix_edge_order_difference(
        graph,
        residue_ids,
        kT,
    )

    print(f"\n{label} EDGE-ORDER AUDIT AT h={h:.1e}")
    print("-" * 62)
    print(f"Exact C             : {exact_result['C']:.12f}")
    print(f"FD C, forward order : {forward['C']:.12f}")
    print(f"FD C, reverse order : {reverse['C']:.12f}")
    print(
        f"FD order difference : "
        f"{reverse['C'] - forward['C']:+.12e}"
    )
    print(
        f"Max |L_reverse-L_forward| at tau=kT: "
        f"{matrix_difference['max_abs']:.12e}"
    )
    print(
        f"Frobenius difference between matrices: "
        f"{matrix_difference['frobenius']:.12e}"
    )
    print(
        f"Forward second-difference numerator: "
        f"{forward['second_numerator']:+.12e}"
    )
    print(
        f"Reverse second-difference numerator: "
        f"{reverse['second_numerator']:+.12e}"
    )


def print_grounding_audit(
    label,
    graph,
    residue_ids,
    kT,
):
    """
    Recompute exact C with several grounded nodes.

    The exact result should be invariant up to numerical roundoff.
    """
    n = len(residue_ids)

    candidate_indices = sorted(
        set([0, n // 2, n - 1])
    )

    results = []

    print(f"\n{label} GROUNDING INVARIANCE AUDIT")
    print("-" * 62)

    for ground_index in candidate_indices:
        result = global_thermo_exact(
            graph,
            residue_ids,
            kT=kT,
            ground_index=ground_index,
        )

        results.append(result["C"])

        residue_label = format_residue_id(
            residue_ids[ground_index]
        )

        print(
            f"Ground index {ground_index:>4} "
            f"(residue {residue_label:>5}): "
            f"C = {result['C']:.12f}"
        )

    spread = max(results) - min(results)

    print(f"Maximum grounding spread: {spread:.12e}")


# ============================================================================
# MAIN PROGRAM
# ============================================================================

def main():
    if KT <= 0:
        raise ValueError("KT must be positive.")

    if CUTOFF <= 0:
        raise ValueError("CUTOFF must be positive.")

    wt_path = ensure_pdb_file(
        WT_PDB_ID,
        WT_PDB_FILE,
        auto_download=AUTO_DOWNLOAD,
    )

    mut_path = ensure_pdb_file(
        MUT_PDB_ID,
        MUT_PDB_FILE,
        auto_download=AUTO_DOWNLOAD,
    )

    print("\nBuilding C-alpha contact graphs...")

    wt_graph, wt_residues, _ = build_ca_graph(
        wt_path,
        chain_id=CHAIN_ID,
        cutoff=CUTOFF,
    )

    mut_graph, mut_residues, _ = build_ca_graph(
        mut_path,
        chain_id=CHAIN_ID,
        cutoff=CUTOFF,
    )

    print_graph_summary(
        "WT 6GOD",
        wt_graph,
        wt_residues,
        CUTOFF,
    )

    print_graph_summary(
        "G12D 6GOF",
        mut_graph,
        mut_residues,
        CUTOFF,
    )

    if wt_residues != mut_residues:
        warnings.warn(
            "WT and mutant residue identifier lists are not identical. "
            "Global thermodynamics can still be calculated, but the graphs "
            "do not contain exactly matching ordered residue sets.",
            RuntimeWarning,
        )

    if not 0 <= GROUND_INDEX < len(wt_residues):
        raise ValueError(
            f"GROUND_INDEX={GROUND_INDEX} is invalid for WT."
        )

    if not 0 <= GROUND_INDEX < len(mut_residues):
        raise ValueError(
            f"GROUND_INDEX={GROUND_INDEX} is invalid for G12D."
        )

    wt_exact = global_thermo_exact(
        wt_graph,
        wt_residues,
        kT=KT,
        ground_index=GROUND_INDEX,
    )

    mut_exact = global_thermo_exact(
        mut_graph,
        mut_residues,
        kT=KT,
        ground_index=GROUND_INDEX,
    )

    print(
        f"\nParameters: chain={CHAIN_ID}, "
        f"cutoff={CUTOFF:.3f} A, kT={KT:.6f}"
    )

    print_exact_table(
        wt_exact,
        mut_exact,
    )

    print_fd_audit(
        wt_graph,
        wt_residues,
        mut_graph,
        mut_residues,
        wt_exact,
        mut_exact,
        kT=KT,
        ground_index=GROUND_INDEX,
        steps=FD_STEPS,
    )

    print_edge_order_audit(
        "WT 6GOD",
        wt_graph,
        wt_residues,
        wt_exact,
        kT=KT,
        h=EDGE_ORDER_AUDIT_H,
        ground_index=GROUND_INDEX,
    )

    print_edge_order_audit(
        "G12D 6GOF",
        mut_graph,
        mut_residues,
        mut_exact,
        kT=KT,
        h=EDGE_ORDER_AUDIT_H,
        ground_index=GROUND_INDEX,
    )

    print_grounding_audit(
        "WT 6GOD",
        wt_graph,
        wt_residues,
        kT=KT,
    )

    print_grounding_audit(
        "G12D 6GOF",
        mut_graph,
        mut_residues,
        kT=KT,
    )

    print("\nINTERPRETATION")
    print("-" * 62)
    print(
        "The EXACT line is the thermodynamic result. "
        "It contains no finite-difference step h."
    )
    print(
        "A finite-difference estimate is reliable only where it agrees "
        "with the exact trace result over a stable range of h."
    )
    print(
        "Strong variation at small h, or sensitivity to edge summation "
        "order, demonstrates floating-point cancellation."
    )
    print(
        "Do not combine the 8.0-A Table-1 audit with a 7.8-A path analysis "
        "without explicitly stating that the two graph constructions differ."
    )

    return {
        "WT": wt_exact,
        "G12D": mut_exact,
    }


if __name__ == "__main__":
    results = main()